# 01. データ準備 ― ウェアハウス・DB・ロールと6テーブル

Amazon Quick × Snowflake ハンズオン（Snowflake編）の **1本目** です。
元教材では SQL ワークシートに貼り付けて一括実行していた部分を、1ステップずつ実行しながら確認します。

| Notebook | 内容 |
|---|---|
| **01（このNotebook）** | ウェアハウス・DB・ロールの作成、S3 から 6 テーブルをロード |
| 02 | Semantic View と Cortex Agent の作成・動作確認 |
| 03 | MCP Server と OAuth の作成、Amazon Quick に入力する値の取得 |

**前提**：Snowflake の **トライアルアカウント**（Enterprise エディション、**AWS** リージョン推奨）。サインアップしたユーザーは ACCOUNTADMIN を持っています。  
**所要時間**：約10分

### Notebook を開く前に（トライアルアカウントの注意）

| 確認すること | 理由 |
|---|---|
| **AI 機能が有効なトライアルか** | 通常のセルフサービストライアルでは Cortex Agent などの AI 機能が**既定で無効**です（`not available for trial accounts`）。イベント用のサインアップ URL で作ったアカウントか、**Admin » Billing でクレジットカードを登録**したアカウントを使ってください（カード登録後も残りの無償クレジットはそのまま使えます）。Step 0 で自動チェックします。 |
| **コンピュートプールは `SYSTEM_COMPUTE_POOL_CPU`** | トライアルではコンピュートプールを新規作成できません。Notebook の接続時に既定の CPU プールを選んでください（GPU は不要です）。 |
| **クエリウェアハウスは既定の `COMPUTE_WH` でよい** | Step 1 で専用ウェアハウスを作り、以降はそちらに切り替えます。 |

> 🖼️ **図の差し込み位置：全体アーキテクチャ**  
> 6テーブル → Semantic View → Cortex Agent → MCP Server → OAuth → Amazon Quick の全体図（このNotebookは左端の「データ」部分）  
> （画像は `images/` に置き、`![全体アーキテクチャ](../images/xxx.png)` で差し込んでください）

## Step 0. セッションの準備

このNotebookで使う名前（DB・スキーマ・ロール・ウェアハウス）を変数にまとめます。
作成系の操作は ACCOUNTADMIN で行い、最後にハンズオン用ロールで参照できることを確認します。

> ウェアハウス `CORTEX_QUICK_HOL_WH` はまだ存在しないため、このセルは Step 1 の後でもう一度実行しても構いません。

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()

DB, SCHEMA = "CORTEX_QUICK_HOL_DB", "CORTEX_QUICK_HOL_SCHEMA"
HOL_ROLE, HOL_WH = "CORTEX_QUICK_HOL_ROLE", "CORTEX_QUICK_HOL_WH"

session.use_role("ACCOUNTADMIN")
info = session.sql("""SELECT CURRENT_USER() AS U, CURRENT_REGION() AS R,
  CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME() AS A""").collect()[0]
hol_user = info["U"]
print("実行ユーザー:", hol_user, "/ ロール:", session.get_current_role())
print("アカウント  :", info["A"], "/ リージョン:", info["R"])

### 事前チェック：AI 機能とクロスリージョン推論

後半（Agent / Quick 連携）で詰まらないよう、**最初に** AI 機能が使えるかを確かめます。

- `CORTEX_ENABLED_CROSS_REGION` は、アカウントのリージョンに必要なモデルがないときに他リージョンで推論する設定です。トライアルのリージョンによっては Agent のモデルが見つからないため、`ANY_REGION` にします（データ自体は移動しません）。
- `AI_COMPLETE` を1回だけ呼び、トライアルの AI 制限にかかっていないかを確認します（数文字の応答なので消費はごくわずかです）。

> ❌ `not available for trial accounts` / `Access denied for trial accounts` が出た場合、この先の Agent は動きません。
> 上の「トライアルアカウントの注意」のとおりクレジットカードを登録し、このセルを再実行してください。

In [ ]:
session.sql("ALTER ACCOUNT SET CORTEX_ENABLED_CROSS_REGION = 'ANY_REGION'").collect()
print("OK: CORTEX_ENABLED_CROSS_REGION = ANY_REGION")

try:
    r = session.sql("SELECT AI_COMPLETE('claude-sonnet-4-5', 'Reply with OK only.') AS R").collect()[0]["R"]
    print("OK: AI 機能が使えます（応答:", str(r).strip()[:20], "）")
except Exception as e:
    msg = str(e)
    if "trial" in msg.lower():
        raise RuntimeError(
            "このトライアルアカウントでは AI 機能が無効です。Admin » Billing » Payment method でクレジットカードを登録してから、"
            "このセルを再実行してください。（イベント用 URL で作ったアカウントでは通常発生しません）") from None
    print("⚠️ AI_COMPLETE の呼び出しに失敗しました。モデル名の問題の可能性があります（Agent の動作は 02 で確認します）:\n", msg[:300])

## Step 1. ウェアハウス・データベース・スキーマ・ロールを作る

| オブジェクト | 名前 | 役割 |
|---|---|---|
| ウェアハウス | `CORTEX_QUICK_HOL_WH` | SQL を実行する計算資源（XSMALL、60秒で自動停止） |
| データベース / スキーマ | `CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA` | テーブルや Agent の置き場所 |
| ロール | `CORTEX_QUICK_HOL_ROLE` | Amazon Quick から接続するときに使う権限のまとまり |

> 💡 **元教材との違い**：元教材では `SNOWFLAKE_USER` を自分のユーザー名に書き換えて実行します。
> ここでは `CURRENT_USER()` で取得した値を使うので、書き換えは不要です。

In [ ]:
stmts = [
    f"CREATE WAREHOUSE IF NOT EXISTS {HOL_WH} WAREHOUSE_SIZE='XSMALL' AUTO_SUSPEND=60 AUTO_RESUME=TRUE INITIALLY_SUSPENDED=TRUE",
    f"CREATE DATABASE IF NOT EXISTS {DB}",
    f"CREATE SCHEMA IF NOT EXISTS {DB}.{SCHEMA}",
    f"CREATE ROLE IF NOT EXISTS {HOL_ROLE}",
    f'GRANT ROLE {HOL_ROLE} TO USER "{hol_user}"',
]
for s in stmts:
    session.sql(s).collect()
    print("OK:", s)

session.use_warehouse(HOL_WH)
session.use_schema(f"{DB}.{SCHEMA}")

### ロールに権限を付ける

Snowflake では「誰が何をできるか」をロールに付けた権限で管理します（RBAC）。
Quick から OAuth でログインしたユーザーも、このロールの権限の範囲でしかデータに触れません。

| 権限 | 意味 |
|---|---|
| `USAGE ON WAREHOUSE` | ウェアハウスで SQL を実行できる |
| `USAGE ON DATABASE` | データベースの中を見られる |
| `ALL ON SCHEMA` | スキーマ内にオブジェクトを作ったり使ったりできる |
| `SELECT ON FUTURE TABLES` | **これから作る**テーブルを自動的に参照できる |
| `SNOWFLAKE.CORTEX_AGENT_USER` | Cortex Agent を呼び出せる（データベースロール） |

In [ ]:
stmts = [
    f"GRANT USAGE ON WAREHOUSE {HOL_WH} TO ROLE {HOL_ROLE}",
    f"GRANT USAGE ON DATABASE {DB} TO ROLE {HOL_ROLE}",
    f"GRANT ALL PRIVILEGES ON SCHEMA {DB}.{SCHEMA} TO ROLE {HOL_ROLE}",
    f"GRANT SELECT ON FUTURE TABLES IN SCHEMA {DB}.{SCHEMA} TO ROLE {HOL_ROLE}",
    f"GRANT DATABASE ROLE SNOWFLAKE.CORTEX_AGENT_USER TO ROLE {HOL_ROLE}",
]
for s in stmts:
    session.sql(s).collect()
    print("OK:", s)

## Step 2. CSV のファイルフォーマットを作る

S3 にある CSV ファイルの読み方（区切り文字・ヘッダー行・NULL の扱い）を定義します。

In [ ]:
%%sql -r create_file_format
CREATE OR REPLACE FILE FORMAT CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.CSV_FORMAT
  TYPE = 'CSV'
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  SKIP_HEADER = 1
  NULL_IF = ('', 'NULL')

## Step 3. 6つのテーブルを作る

営業オペレーションのサンプルデータです。取引（ファクト）を中心に、顧客・製品・営業担当が紐づきます。

| テーブル | 内容 | 主なカラム |
|---|---|---|
| `SALES_TRANSACTIONS_FACT_TABLE` | 受注した取引（ファクト） | 地域、金額、割引率、営業期間 |
| `CUSTOMER_DIMENSION_TABLE` | 顧客 | 業種、規模、LTV、解約リスク |
| `PRODUCT_CATALOG_TABLE` | 製品カタログ | カテゴリ、ティア、定価、更新率 |
| `SALES_REP_PERFORMANCE_TABLE` | 営業担当の実績 | テリトリー、目標、達成率、勝率 |
| `SALES_PIPELINE_OPPORTUNITIES` | 商談パイプライン | ステージ、見込み金額、競合有無 |
| `MARKETING_ATTRIBUTION_TABLE` | マーケティング施策 | キャンペーン、チャネル、リードスコア |

> 🖼️ **図の差し込み位置：テーブル関係図（ER図）**  
> 取引テーブルが CUSTOMER_ID / PRODUCT_SKU / SALES_REP_ID で顧客・製品・営業担当につながる図。パイプラインとマーケティングは独立  
> （画像は `images/` に置き、`![テーブル関係図（ER図）](../images/xxx.png)` で差し込んでください）

In [ ]:
%%sql -r create_tables
CREATE OR REPLACE TABLE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE (
  TRANSACTION_ID VARCHAR, TRANSACTION_DATE DATE, CUSTOMER_ID VARCHAR, PRODUCT_SKU VARCHAR,
  SALES_REP_ID VARCHAR, REGION VARCHAR, DEAL_SIZE_USD NUMBER(12,2), QUANTITY NUMBER,
  DISCOUNT_PERCENTAGE NUMBER(5,2), SALES_CYCLE_DAYS NUMBER, LEAD_SOURCE VARCHAR,
  CLOSE_PROBABILITY_AT_START NUMBER(5,2), FORECAST_CLOSE_DATE DATE, ACTUAL_CLOSE_DATE DATE
);
CREATE OR REPLACE TABLE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.CUSTOMER_DIMENSION_TABLE (
  CUSTOMER_ID VARCHAR, COMPANY_NAME VARCHAR, INDUSTRY VARCHAR, COMPANY_SIZE VARCHAR,
  ANNUAL_REVENUE_BAND VARCHAR, FIRST_PURCHASE_DATE DATE, CUSTOMER_LIFETIME_VALUE NUMBER(12,2),
  CHURN_RISK_SCORE NUMBER(5,2), TOTAL_CONTRACTS_COUNT NUMBER, EXPANSION_POTENTIAL VARCHAR
);
CREATE OR REPLACE TABLE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_CATALOG_TABLE (
  PRODUCT_SKU VARCHAR, PRODUCT_NAME VARCHAR, PRODUCT_CATEGORY VARCHAR, PRODUCT_FEATURE VARCHAR,
  PRODUCT_TIER VARCHAR, BILLING_FREQUENCY VARCHAR, LIST_PRICE_USD NUMBER(12,2),
  AVERAGE_CONTRACT_LENGTH_MONTHS NUMBER, RENEWAL_RATE_PCT NUMBER(5,2), UPSELL_RATE_PCT NUMBER(5,2),
  TYPICAL_DISCOUNT_RANGE VARCHAR
);
CREATE OR REPLACE TABLE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_REP_PERFORMANCE_TABLE (
  SALES_REP_ID VARCHAR, REP_NAME VARCHAR, TERRITORY VARCHAR, HIRE_DATE DATE, QUOTA_USD NUMBER(12,2),
  QUOTA_ATTAINMENT_PCT NUMBER(5,2), AVG_DEAL_SIZE NUMBER(12,2), WIN_RATE_PCT NUMBER(5,2),
  AVG_SALES_CYCLE_DAYS NUMBER(5,2), PIPELINE_COVERAGE_RATIO NUMBER(5,2), ACTIVE_OPPORTUNITIES_COUNT NUMBER
);
CREATE OR REPLACE TABLE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_PIPELINE_OPPORTUNITIES (
  OPPORTUNITY_ID VARCHAR, OPPORTUNITY_NAME VARCHAR, CURRENT_STAGE VARCHAR, ESTIMATED_VALUE NUMBER(12,2),
  WEIGHTED_FORECAST_VALUE NUMBER(12,2), CLOSE_PROBABILITY NUMBER(5,2), CREATED_DATE DATE,
  EXPECTED_CLOSE_DATE DATE, STAGE_ENTRY_DATE DATE, LAST_ACTIVITY_DATE DATE, DAYS_IN_CURRENT_STAGE NUMBER,
  COMPETITOR_PRESENT BOOLEAN, DECISION_MAKER_ENGAGED BOOLEAN, ACTIVITY_COUNT_LAST_30_DAYS NUMBER
);
CREATE OR REPLACE TABLE CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.MARKETING_ATTRIBUTION_TABLE (
  LEAD_ID VARCHAR, LEAD_SOURCE_DETAIL VARCHAR, CAMPAIGN_NAME VARCHAR, FIRST_TOUCH_CHANNEL VARCHAR,
  LAST_TOUCH_CHANNEL VARCHAR, LEAD_SCORE NUMBER(5,2), LEAD_TO_OPPORTUNITY_DAYS NUMBER,
  MARKETING_SPEND_ALLOCATED NUMBER(12,2)
)

## Step 4. S3 からデータをロードする

`COPY INTO` で、公開 S3 バケットの CSV を各テーブルに読み込みます。認証情報は不要です（トライアルアカウントでもそのまま動きます）。
結果の `status` が `LOADED`、`rows_loaded` が 200 になっていれば成功です。

> 💡 同じセルを2回実行すると `LOAD_SKIPPED`（読み込み済み）になります。テーブルを作り直した（Step 3 を再実行した）場合は再度 `LOADED` になります。

In [ ]:
S3 = "s3://snowflake-corp-se-workshop/summit26-coco-mcp-aws-quick-hol/csvs"
tables = ["SALES_TRANSACTIONS_FACT_TABLE", "CUSTOMER_DIMENSION_TABLE", "PRODUCT_CATALOG_TABLE",
          "SALES_REP_PERFORMANCE_TABLE", "SALES_PIPELINE_OPPORTUNITIES", "MARKETING_ATTRIBUTION_TABLE"]
for t in tables:
    r = session.sql(f"COPY INTO {DB}.{SCHEMA}.{t} FROM '{S3}/{t.lower()}.csv' FILE_FORMAT = {DB}.{SCHEMA}.CSV_FORMAT").collect()[0]
    print(f"{t:<32} {r['status']:<8} rows_loaded={r['rows_loaded']}")
    assert r["status"] in ("LOADED", "LOAD_SKIPPED"), f"{t} のロードに失敗しました: {r}"

### 既存テーブルにも SELECT 権限を付ける

Step 1 の `FUTURE TABLES` の付与は、**付与した後に作ったテーブル**に効きます。
念のため、作成済みのテーブルにも明示的に付与します。

> 💡 **元教材との違い**：元教材ではテーブル作成前に `GRANT SELECT ON ALL TABLES` を実行しているため、
> その時点で存在しないテーブルには権限が付きません。ここではロード後に付与しています。

In [ ]:
session.sql(f"GRANT SELECT ON ALL TABLES IN SCHEMA {DB}.{SCHEMA} TO ROLE {HOL_ROLE}").collect()
print("OK: 既存テーブルへの SELECT を付与しました")

## Step 5. 確認する

6テーブルすべてに **200件** ずつ入っていれば準備完了です。

In [ ]:
%%sql -r row_counts
SELECT 'SALES_TRANSACTIONS_FACT_TABLE' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE
UNION ALL SELECT 'CUSTOMER_DIMENSION_TABLE',     COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.CUSTOMER_DIMENSION_TABLE
UNION ALL SELECT 'PRODUCT_CATALOG_TABLE',        COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_CATALOG_TABLE
UNION ALL SELECT 'SALES_REP_PERFORMANCE_TABLE',  COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_REP_PERFORMANCE_TABLE
UNION ALL SELECT 'SALES_PIPELINE_OPPORTUNITIES', COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_PIPELINE_OPPORTUNITIES
UNION ALL SELECT 'MARKETING_ATTRIBUTION_TABLE',  COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.MARKETING_ATTRIBUTION_TABLE

### テーブル同士がつながるか確認する

Semantic View では、取引テーブルを顧客・製品・営業担当とキーで結合します。実際に一致する行数を見ておきます。

In [ ]:
%%sql -r join_check
SELECT
  (SELECT COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE t JOIN CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.CUSTOMER_DIMENSION_TABLE c ON t.CUSTOMER_ID = c.CUSTOMER_ID)     AS TXN_TO_CUSTOMER,
  (SELECT COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE t JOIN CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_REP_PERFORMANCE_TABLE r ON t.SALES_REP_ID = r.SALES_REP_ID) AS TXN_TO_REP,
  (SELECT COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE t JOIN CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_CATALOG_TABLE p ON t.PRODUCT_SKU = p.PRODUCT_SKU)        AS TXN_TO_PRODUCT_CATALOG

> ⚠️ **元データの問題：取引と製品カタログが結合できない**
> 顧客・営業担当は 200 件すべて一致しますが、**製品カタログとは 0 件**です。
> 取引の `PRODUCT_SKU` は `ENT-ANNUAL` のような「ティア＋請求サイクル」（6種類）、製品カタログは `ENT-ANN-011` のような個別製品（200種類）で、値の形式が違うためです（元教材のデータのままです）。
> このままだと「ティア別の売上は？」という Semantic View の検証済みクエリも 0 件になってしまいます。

## Step 6. 取引と製品をつなぐ「プラン」ビューを作る

取引の SKU を分解すると、ティア（`ENT` → Enterprise など）と請求サイクル（`ANNUAL` → Annual）が取り出せます。
製品カタログにも同じ2つのカラムがあるので、**プラン（ティア×請求サイクル）単位**のビューを作れば両者をつなげられます。

| ビューのカラム | 意味 |
|---|---|
| `PRODUCT_SKU` | 取引と同じ SKU（`ENT-ANNUAL` など）→ **取引と 1対多 で結合できる** |
| `PRODUCT_TIER` / `BILLING_FREQUENCY` | Enterprise / Professional / Starter、Annual / Monthly |
| `CATALOG_PRODUCT_COUNT` / `AVG_LIST_PRICE_USD` / `AVG_RENEWAL_RATE_PCT` / `AVG_UPSELL_RATE_PCT` / `AVG_CONTRACT_LENGTH_MONTHS` | そのプランに属するカタログ製品の数・平均定価・平均更新率・平均アップセル率・平均契約期間 |

> 💡 **元教材との違い**：Semantic View の製品テーブルをこのビューに差し替えています。これで「ティア別」「請求サイクル別」の売上に答えられます。
> 個別製品やカテゴリ単位の売上は、元データに情報がないため答えられません（製品カタログ単体の質問には答えられます）。

> 🖼️ **図の差し込み位置：プランビューによる結合**  
> 取引 PRODUCT_SKU（ENT-ANNUAL）→ PRODUCT_PLAN_VIEW（Enterprise × Annual）← 製品カタログ（ティア＋請求サイクルで集約）の図  
> （画像は `images/` に置き、`![プランビューによる結合](../images/xxx.png)` で差し込んでください）

In [ ]:
%%sql -r create_product_plan_view
CREATE OR REPLACE VIEW CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_PLAN_VIEW
  COMMENT = 'Plan (tier x billing) level product view joinable to SALES_TRANSACTIONS_FACT_TABLE.PRODUCT_SKU'
AS
WITH plans AS (
  SELECT DISTINCT PRODUCT_SKU,
    CASE SPLIT_PART(PRODUCT_SKU, '-', 1)
      WHEN 'ENT' THEN 'Enterprise' WHEN 'PRO' THEN 'Professional' WHEN 'STARTER' THEN 'Starter'
    END AS PRODUCT_TIER,
    INITCAP(SPLIT_PART(PRODUCT_SKU, '-', 2)) AS BILLING_FREQUENCY
  FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE
)
SELECT p.PRODUCT_SKU, p.PRODUCT_TIER, p.BILLING_FREQUENCY,
  COUNT(c.PRODUCT_SKU)              AS CATALOG_PRODUCT_COUNT,
  ROUND(AVG(c.LIST_PRICE_USD), 2)   AS AVG_LIST_PRICE_USD,
  ROUND(AVG(c.RENEWAL_RATE_PCT), 2) AS AVG_RENEWAL_RATE_PCT,
  ROUND(AVG(c.UPSELL_RATE_PCT), 2)  AS AVG_UPSELL_RATE_PCT,
  ROUND(AVG(c.AVERAGE_CONTRACT_LENGTH_MONTHS), 1) AS AVG_CONTRACT_LENGTH_MONTHS
FROM plans p
LEFT JOIN CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_CATALOG_TABLE c
  ON c.PRODUCT_TIER = p.PRODUCT_TIER AND c.BILLING_FREQUENCY = p.BILLING_FREQUENCY
GROUP BY 1, 2, 3

In [ ]:
session.sql(f"GRANT SELECT ON VIEW {DB}.{SCHEMA}.PRODUCT_PLAN_VIEW TO ROLE {HOL_ROLE}").collect()
print("OK: PRODUCT_PLAN_VIEW への SELECT を付与しました")

6行（3ティア × 2請求サイクル）が表示され、`CATALOG_PRODUCT_COUNT` がすべて 1 以上なら成功です。
取引 200 件すべてがプランに結合できることも確認します（`TXN_TO_PLAN = 200`）。

In [ ]:
%%sql -r check_product_plan_view
SELECT v.*,
  (SELECT COUNT(*) FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.SALES_TRANSACTIONS_FACT_TABLE t
     JOIN CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_PLAN_VIEW p ON t.PRODUCT_SKU = p.PRODUCT_SKU) AS TXN_TO_PLAN
FROM CORTEX_QUICK_HOL_DB.CORTEX_QUICK_HOL_SCHEMA.PRODUCT_PLAN_VIEW v
ORDER BY v.PRODUCT_SKU

✅ 01 はここまでです。次は `02_semantic_view_and_agent.ipynb` に進みます。